# 02 — Results Visualization
Compare MTL vs YOLO baselines: training curves, mIoU, PCK, and qualitative predictions.

In [1]:
import json
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys; sys.path.append('..')
from config import DATA_CFG, MODEL_CFG, TRAIN_CFG
from src.models.mtl_net import HybridMTLNet

DEVICE = torch.device(TRAIN_CFG.device)
WEIGHTS = '../models/checkpoints/model_final.pth'

model = HybridMTLNet(MODEL_CFG.num_classes, MODEL_CFG.num_keypoints).to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()
print('Model loaded')

FileNotFoundError: [Errno 2] No such file or directory: '../models/checkpoints/model_final.pth'

In [ ]:
# Plot training curves
curves_img = cv2.imread('../models/checkpoints/training_curves.png')
if curves_img is not None:
    plt.figure(figsize=(16, 5))
    plt.imshow(cv2.cvtColor(curves_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Curves')
    plt.show()
else:
    print('Run training first to generate training_curves.png')

In [ ]:
# Qualitative: run on test samples and show predictions
from src.data.dataset import MouseMTLDataset

with open('../data/dataset.json') as f:
    data = json.load(f)

test_items = data['test'][:4]  # show 4 samples
KP_COLORS = [(255, 0, 0), (255, 165, 0), (0, 0, 255)]
tw, th = DATA_CFG.target_size

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, item in enumerate(test_items):
    img_bgr = cv2.imread(item['image_path'])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    resized = cv2.resize(img_rgb, (tw, th))
    tensor = ((torch.from_numpy(resized.transpose(2,0,1)).float()/255.0 - mean) / std).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_seg, pred_pose = model(tensor)

    # -- GT row --
    gt_overlay = img_rgb.copy()
    combined = np.zeros((h, w), dtype=np.uint8)
    for m_path in item['mask_paths']:
        m = cv2.imread(m_path, cv2.IMREAD_GRAYSCALE)
        if m is not None:
            combined = np.maximum(combined, m)
    gt_overlay[combined > 0] = (gt_overlay[combined > 0] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
    for mouse_kps in item['all_keypoints']:
        for k, (x, y) in enumerate(mouse_kps):
            if x > 0 and y > 0:
                cv2.circle(gt_overlay, (int(x), int(y)), 6, KP_COLORS[k], -1)
    axes[0, col].imshow(gt_overlay); axes[0, col].set_title(f'GT [{col}]'); axes[0, col].axis('off')

    # -- Pred row --
    seg_mask = torch.argmax(pred_seg, 1).squeeze().cpu().numpy().astype(np.uint8)
    seg_mask = cv2.resize(seg_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    pred_overlay = img_rgb.copy()
    pred_overlay[seg_mask == 1] = (pred_overlay[seg_mask == 1] * 0.5 + np.array([0, 200, 255]) * 0.5).astype(np.uint8)
    pose_np = pred_pose.squeeze().cpu().numpy()
    for k in range(MODEL_CFG.num_keypoints):
        _, mv, _, loc = cv2.minMaxLoc(pose_np[k])
        if mv > 0.1:
            px, py = int(loc[0]/tw*w), int(loc[1]/th*h)
            cv2.circle(pred_overlay, (px, py), 6, KP_COLORS[k], -1)
    axes[1, col].imshow(pred_overlay); axes[1, col].set_title(f'Pred [{col}]'); axes[1, col].axis('off')

axes[0, 0].set_ylabel('Ground Truth', fontsize=11)
axes[1, 0].set_ylabel('Prediction', fontsize=11)
plt.suptitle('MTL Predictions (seg=cyan, kp: nose=red, shoulder=orange, tail=blue)', fontsize=12)
plt.tight_layout()
plt.show()